# Exploratory hippocampal LFP and behavior

Visualize Mitt’s raw traces, trial-aligned averages, and behavior summaries. The notebook reuses the validated raw-trial reader. For visualization only, the averages use a 30 Hz low-pass and a pre-poke baseline, not the poster feature pipeline.

Exploratory companion to the poster, not an input to the reviewed results. See the [README](../README.md#exploratory-notebooks) for the environment and data requirements.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
repo = next((p for p in (here, *here.parents) if (p / 'src/data_processing').is_dir()), None)
if repo is None:
    raise FileNotFoundError('Start Jupyter from the repository or its notebooks directory.')
sys.path.insert(0, str(repo / 'src'))
session = '080718_mitt'
raw = repo / 'data' / 'raw_data' / session
from scipy.signal import butter, sosfiltfilt
from data_processing.vendor.events import load_behavior
from data_processing.vendor.io import metadata, rows

time, raw_trials, segments, _, _, _ = load_behavior(raw / f'{session}_bvr.npz')
trials = pd.DataFrame(t for t in raw_trials if not t['reasons'])
trials['odor'] = trials['odor'].map(dict(enumerate('ABCDE')))
trials['position'] += 1
trials['sequence'] = trials['inseq'].map({1: 'InSeq', 0: 'OutSeq'})
trials['hold_s'] = trials['poke_out_s'] - trials['poke_in_s']
print(f'{len(trials)}/{len(raw_trials)} onsets have unambiguous behavior and poke boundaries.')
display(trials.groupby('odor')['hold_s'].agg(['size', 'mean', 'median']).round(3))

## Poke-aligned LFP

Each plotted window and its filter padding must fit within a single recording segment. Baselines use the pre-poke samples; samples at and after withdrawal are omitted from the averages. Support consequently varies over time. The same filtered trial tensor feeds both plots.

In [ ]:
pre, post, pad = 500, 1200, 500
bounds = np.asarray(segments)[trials['segment'].to_numpy()]
start = trials['poke_in'].to_numpy()
keep = (start - pre - pad >= bounds[:, 0]) & (start + post + pad <= bounds[:, 1])
panel = trials.loc[keep].reset_index(drop=True)
if panel.empty:
    raise ValueError('No trials have complete padded plotting windows.')
start = panel['poke_in'].to_numpy()
index = start[:, None] + np.arange(-pre - pad, post + pad)
lfp_path = raw / f'{session}_lfp.npz'
meta = metadata(lfp_path)
if meta['shape'][1] != len(time):
    raise ValueError('LFP and behavior sample counts differ.')
channels = meta['keys']
voltage = np.stack([row[index] for row in rows(lfp_path)], axis=-1)
sos = butter(4, 30, btype='lowpass', fs=1000, output='sos')
filtered = sosfiltfilt(sos, voltage, axis=1)[:, pad:-pad]
filtered -= filtered[:, :pre].mean(axis=1, keepdims=True)
relative = np.arange(-pre, post)
held = panel['poke_out'].to_numpy() - start
filtered[relative[None, :] >= held[:, None]] = np.nan
t = relative / 1000

def plot_mean(ax, traces, label, color):
    if len(traces) == 0:
        return
    support = np.isfinite(traces).sum(axis=0)
    mean = np.divide(np.nansum(traces, axis=0), support,
                     out=np.full(len(t), np.nan), where=support > 0)
    ss = np.nansum((traces - mean) ** 2, axis=0)
    sem = np.sqrt(np.divide(ss, support * (support - 1),
                            out=np.full(len(t), np.nan), where=support > 1))
    ax.plot(t, mean, color=color, label=f'{label} (n={len(traces)})')
    ax.fill_between(t, mean - sem, mean + sem, color=color, alpha=.15, linewidth=0)

odors = 'ABCDE'
pool_incorrect = False  # True reproduces the pooled-incorrect view from the lit-review branch.
conditions = [('InSeq correct', 'InSeq', 1, '#1a9850'),
              ('OutSeq correct', 'OutSeq', 1, '#4393c3')]
if pool_incorrect:
    conditions.append(('Incorrect (either)', None, 0, '#d73027'))
else:
    conditions.extend([('InSeq incorrect', 'InSeq', 0, '#f46d43'),
                       ('OutSeq incorrect', 'OutSeq', 0, '#762a83')])
fig, axes = plt.subplots(1, 5, figsize=(17, 4), sharex=True, sharey=True, layout='constrained')
pooled = filtered.mean(axis=2)
for ax, item in zip(axes, odors, strict=True):
    for label, seq, correct, color in conditions:
        selected = panel['odor'].eq(item) & panel['correct'].eq(correct)
        if seq is not None:
            selected &= panel['sequence'].eq(seq)
        plot_mean(ax, pooled[selected], label, color)
    ax.axvline(0, color='gray', linestyle='--', linewidth=.8)
    ax.set_title(f'Odor {item}')
    ax.set_xlabel('Time from poke entry (s)')
    if ax.lines[:-1]:
        ax.legend(fontsize=7)
axes[0].set_ylabel('Mean baseline-subtracted LFP across channels')
plt.show()

### Individual channels, correct InSeq trials

In [ ]:
ncols = 6
nrows = (len(channels) + ncols - 1) // ncols
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
fig, grid = plt.subplots(nrows, ncols, figsize=(17, 2.6 * nrows), sharex=True, sharey=True,
                         squeeze=False, layout='constrained')
for j, ax in enumerate(grid.flat):
    if j >= len(channels):
        ax.axis('off')
        continue
    for item, color in zip(odors, colors[:len(odors)], strict=True):
        selected = panel['odor'].eq(item) & panel['inseq'].eq(1) & panel['correct'].eq(1)
        plot_mean(ax, filtered[selected, :, j], f'Odor {item}', color)
    ax.axvline(0, color='gray', linestyle='--', linewidth=.8)
    ax.set_title(channels[j].split('_')[0])
if grid[0, 0].lines[:-1]:
    grid[0, 0].legend(fontsize=7)
fig.supxlabel('Time from poke entry (s)')
fig.supylabel('Baseline-subtracted LFP')
plt.show()

### One unfiltered trial

Traces are centered and vertically offset for display. Lines mark recorded events, not an inferred stimulus offset or decision time.

In [ ]:
i = panel.index[panel['odor'].eq('A') & panel['inseq'].eq(1) & panel['correct'].eq(1)][0]
trial = panel.loc[i]
trace = voltage[i, pad:-pad].copy()
trace -= trace.mean(axis=0)
gap = 4 * np.median(trace.std(axis=0))
offsets = -gap * np.arange(len(channels))
fig, ax = plt.subplots(figsize=(11, 8), layout='constrained')
ax.plot(t, trace + offsets, color='#444444', linewidth=.65)
ax.set_yticks(offsets, channels)
for event, label in [('poke_in', 'Poke entry'), ('onset', 'Odor onset'), ('poke_out', 'Withdrawal')]:
    x = (trial[event] - trial['poke_in']) / 1000
    if t[0] <= x <= t[-1]:
        ax.axvline(x, linestyle='--', label=label)
ax.set(xlabel='Time from poke entry (s)', title=f'Odor {trial.odor} · raw trial {trial.trial_id}')
ax.legend()
plt.show()

## Correctness by position and session time

These summaries use all unambiguous behavior trials, not just the plotting windows. The 10-minute bins count recorded samples and exclude gaps in the clock.

In [ ]:
display(trials.pivot_table(index='position', columns='sequence', values='correct',
                           aggfunc=['mean', 'count']).round(3))
minutes = trials['poke_in'] / (1000 * 60)
blocks = trials.assign(block=(minutes // 10).astype(int))
counts = blocks.groupby(['block', 'sequence'])['correct'].agg(correct='sum', total='size')
proportions = counts['correct'].div(counts['total']).unstack('sequence')
ax = proportions.plot.bar(figsize=(10, 4), ylim=(0, 1), rot=0, ylabel='Proportion correct')
ax.set_xticklabels([f'{10*b}–{10*(b+1)}' for b in proportions.index])
ax.set_xlabel('Recorded minutes since session start (gaps excluded)')
for bars, sequence in zip(ax.containers, proportions.columns, strict=True):
    label = counts.xs(sequence, level='sequence').reindex(proportions.index)
    ax.bar_label(bars, labels=[f'{int(c)}/{int(n)}' if pd.notna(n) else ''
                               for c, n in zip(label.correct, label.total, strict=True)], padding=2)
ax.set_ylim(0, 1.12)
display(counts)
plt.show()